# 09 Model Experiment

This notebook is a sandbox for testing alternative model and threshold choices without changing the final pipeline artifacts.

Goal:
- keep the existing preprocessing logic
- train experimental models separately
- compare metrics and threshold trade-offs
- decide later whether any experiment should replace the current final model

Important: do not overwrite files in `models/` from this notebook unless the experiment has been reviewed and accepted.

## Experiment Rules

- Use the same train/test split logic as the project.
- Fit preprocessing only on the training split.
- Evaluate on the held-out test split.
- Compare AUROC, AUPRC, precision, recall, F1, FP, FN, TP, and TN.
- Treat threshold selection as a clinical trade-off, not a default assumption.
- Save experimental outputs under a separate path only if needed, for example `reports/01_modeling/experiments/`.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

from src.preprocessing import ICUPreprocessor

pd.set_option('display.max_columns', 100)

DATA_PATH = ROOT / 'data/raw/training_v2.csv'
EXPERIMENT_DIR = ROOT / 'reports/01_modeling/experiments'
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH

## Load Data and Recreate Split

This section should mirror the project split so model comparisons are fair.

In [ ]:
from sklearn.model_selection import train_test_split

raw = pd.read_csv(DATA_PATH)

raw_train, raw_test = train_test_split(
    raw,
    test_size=0.2,
    random_state=42,
    stratify=raw['hospital_death'],
)

y_train = raw_train['hospital_death'].astype(int)
y_test = raw_test['hospital_death'].astype(int)

print('raw_train:', raw_train.shape)
print('raw_test :', raw_test.shape)
print('train death rate:', round(y_train.mean(), 4))
print('test death rate :', round(y_test.mean(), 4))

## Apply Existing Preprocessing Logic

The preprocessor is fit on `raw_train` only, then applied to both train and test.

In [ ]:
preprocessor = ICUPreprocessor()
preprocessor.fit(raw_train)

X_train = preprocessor.transform(raw_train)
X_test = preprocessor.transform(raw_test)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)

## Evaluation Helpers

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


def evaluate_probabilities(y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        'threshold': threshold,
        'AUROC': roc_auc_score(y_true, y_proba),
        'AUPRC': average_precision_score(y_true, y_proba),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'TN': tn,
        'FP': fp,
        'FN': fn,
        'TP': tp,
    }


def threshold_sweep(y_true, y_proba, thresholds=None):
    if thresholds is None:
        thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)

    return pd.DataFrame(
        [evaluate_probabilities(y_true, y_proba, threshold=t) for t in thresholds]
    )

## Experimental Model Area

Paste or adapt the experimental model code here. Keep outputs separate from the final model artifacts.

In [ ]:
# Example placeholder: replace with the model you want to test.
# from lightgbm import LGBMClassifier
#
# experimental_model = LGBMClassifier(
#     n_estimators=500,
#     learning_rate=0.03,
#     random_state=42,
#     class_weight='balanced',
# )
# experimental_model.fit(X_train, y_train)
# y_proba = experimental_model.predict_proba(X_test)[:, 1]
#
# evaluate_probabilities(y_test, y_proba, threshold=0.5)

## Threshold Review

Use this section to compare threshold choices. In this clinical risk setting, lower thresholds may improve recall and reduce false negatives, but they usually increase false positives.

In [ ]:
# After producing y_proba, run:
# sweep_df = threshold_sweep(y_test, y_proba)
# sweep_df.sort_values('F1', ascending=False).head(10)
# sweep_df.to_csv(EXPERIMENT_DIR / 'threshold_sweep_experiment.csv', index=False)